# Vision AI — Real-Time Multi-Modal Threat Detection (Google Colab)

Loads **every detection model into GPU VRAM**, opens the **browser webcam**, and runs
**real-time** inference: person detection + ByteTrack tracking, skeletal pose ("stick")
analysis, weapon detection, fire/smoke hazard detection, ST-GCN action + violence
recognition, and facial emotion / crying. Bounding boxes and skeletons are drawn in the
frame; **active alerts and the live THREAT level are shown in a panel at the top-right**.

No VLM / narrative and no audio here — this is the *weights + maths + camera* live path.
Engineered for a **smooth, low-jitter overlay** (see the anti-jitter notes below).

### 1. Models, Classes & Anti-Jitter Design

| Component | Weights | Classes | Role |
| :--- | :--- | :--- | :--- |
| Person + Tracking | `rtdetr-l.pt` + ByteTrack | `person` (COCO 0) | Body boxes & persistent IDs |
| Skeletal Pose | `yolo11x-pose.pt` | 17 COCO joints | One **full-frame** pose pass → skeletons + ST-GCN |
| Weapon | `Fine Tuned Weights/best.pt` | `pistol`→firearm, `knife` | Lookalikes (billete/monedero/smartphone/tarjeta) dropped |
| Fire / Smoke | `firedetect-11s.pt` | `fire`, `smoke` | YOLO conf + HSV physics composite gate |
| Violence | `Fine Tuned Weights/violence_stgcn_best.pt` | `non_violence`, `violence` | ST-GCN over top-2 skeletons (32 frames) |
| Home Action | `Fine Tuned Weights/home_action_stgcn_best_macrof1.pt` | falling_down, lying_on_floor, … | ST-GCN per track; alerts on fall/lying |
| Emotion | `Fine Tuned Weights/emotion_fer_best.pt` | angry…surprise | Stress index + EAR crying |

**Anti-jitter strategy (the live goal):**
1. **Single full-frame pose call** per frame (not N per-person crops) — keypoints matched to tracks by box containment.
2. **Strided heavy models** — weapon / fire / face+emotion / ST-GCN run every few frames; their results are **cached and re-drawn** between runs (persistent overlays, no flicker).
3. **EMA smoothing** of drawn box corners and keypoints per track (`v = α·v_prev + (1-α)·v_now`).
4. **Skeleton confidence gate (>0.35)** + stable track IDs keep the stick figure steady; **FP32 pose** avoids degenerate empty detections.
5. **Alert TTL** — each alert persists a few frames so the top-right panel doesn't strobe.

### 2. Math & Heuristic Reference

**EAR / crying** (InsightFace 106-pt): $EAR=\frac{\lVert p_2-p_6\rVert+\lVert p_3-p_5\rVert}{2\lVert p_1-p_4\rVert}$, $\;P(\text{cry})=\text{clip}\!\left(\frac{0.28-EAR}{0.11},0,1\right)$.

**Fall (spine angle)**: $\theta=\lvert\deg(\text{atan2}(\lvert\Delta x\rvert,\lvert\Delta y\rvert))\rvert$ from shoulder→hip centroids; alert if $\theta>65^\circ$ or torso height $<0.5\times$ shoulder width.

**Immobility / lying**: over 15 frames, $\text{Var}(K_{x,y})/\max(10,W_s)<0.05$ on both axes ⇒ unresponsive.

**Stress**: $1.0P(\text{fear})+0.7P(\text{angry})+0.6P(\text{sad})+0.4P(\text{disgust})+0.2P(\text{surprise})$.

**Skeleton normalization for ST-GCN**: $x_n=(x/W-0.5)\cdot2,\; y_n=(y/H-0.5)\cdot2$; input $(1,3,32,17,M)$.

**Threat weights**: fire 1.0, weapon 0.90, fall/lying 0.90, violence 0.85, aggressive_guard 0.80, overcrowd 0.50, cry 0.35, stress 0.45. Levels: GREEN ≤0.30, YELLOW ≤0.60, ORANGE ≤0.80, RED >0.80.

In [ ]:
# Cell 1 — Mount Google Drive and verify the weights directory
from google.colab import drive
import os

drive.mount('/content/drive')

WEIGHTS_DIR = "/content/drive/MyDrive/VisionAI/models/weights"
print(f"Checking weights at: {WEIGHTS_DIR}")
if os.path.isdir(WEIGHTS_DIR):
    print("Found. Files:")
    for f in sorted(os.listdir(WEIGHTS_DIR)):
        print("  -", f)
else:
    print("WARNING: weights directory not found. Adjust WEIGHTS_DIR to your Drive path.")

In [ ]:
# Cell 2 — Install dependencies (vision-only; no audio/VLM on the live path)
!pip install -q ultralytics supervision timm insightface faiss-cpu onnxruntime-gpu
print("Dependencies installed.")

In [ ]:
# Cell 3 — Shared ST-GCN architecture (COCO-17 graph), identical to the training notebooks
import torch
import torch.nn as nn
import numpy as np

COCO_EDGES = [(0, 1), (0, 2), (1, 3), (2, 4), (0, 5), (0, 6), (5, 6), (5, 7), (7, 9),
              (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]
NUM_JOINTS = 17

def build_adjacency():
    A = np.zeros((NUM_JOINTS, NUM_JOINTS), dtype=np.float32)
    for i, j in COCO_EDGES:
        A[i, j] = 1.0
        A[j, i] = 1.0
    A += np.eye(NUM_JOINTS, dtype=np.float32)
    deg = A.sum(1)
    Dinv = np.diag(1.0 / np.maximum(deg, 1e-6)).astype(np.float32)
    return (Dinv @ A).astype(np.float32)

class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, residual=True):
        super().__init__()
        self.register_buffer("A", torch.tensor(A))
        self.edge_imp = nn.Parameter(torch.ones_like(self.A))
        self.gcn = nn.Conv2d(cin, cout, 1)
        self.tcn = nn.Sequential(nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)), nn.BatchNorm2d(cout))
        if not residual:
            self.res = None
        elif cin == cout and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = 0 if self.res is None else self.res(x)
        x = self.gcn(x)
        x = torch.einsum("nctv,vw->nctw", x, self.A * self.edge_imp)
        x = self.tcn(x)
        return self.relu(x + res)

class STGCN(nn.Module):
    def __init__(self, in_ch=3, num_classes=2):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(in_ch * NUM_JOINTS)
        self.layers = nn.ModuleList([
            STGCNBlock(in_ch, 64, A, residual=False), STGCNBlock(64, 64, A),
            STGCNBlock(64, 128, A, stride=2), STGCNBlock(128, 128, A),
            STGCNBlock(128, 256, A, stride=2), STGCNBlock(256, 256, A)])
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 1, 3, 2).contiguous().view(N * M, C * V, T)
        x = self.data_bn(x)
        x = x.view(N * M, C, V, T).permute(0, 1, 3, 2).contiguous()
        for layer in self.layers:
            x = layer(x)
        x = x.mean(dim=[2, 3]).view(N, M, -1).mean(dim=1)
        return self.fc(self.drop(x))

print("ST-GCN definition loaded.")

In [ ]:
# Cell 4 — Load every model into GPU VRAM (once)
import cv2
import timm
from ultralytics import YOLO
from insightface.app import FaceAnalysis

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

print("Loading RT-DETR-L person detector ...")
person_model = YOLO(f"{WEIGHTS_DIR}/rtdetr-l.pt").to(device)

print("Loading YOLO11x-Pose skeleton extractor ...")
pose_model = YOLO(f"{WEIGHTS_DIR}/yolo11x-pose.pt").to(device)

print("Loading fine-tuned YOLO11n weapon detector ...")
weapon_model = YOLO(f"{WEIGHTS_DIR}/Fine Tuned Weights/best.pt").to(device)

print("Loading YOLO11s fire/smoke detector ...")
fire_model = YOLO(f"{WEIGHTS_DIR}/firedetect-11s.pt").to(device)

print("Loading ST-GCN violence classifier ...")
violence_ckpt = torch.load(f"{WEIGHTS_DIR}/Fine Tuned Weights/violence_stgcn_best.pt",
                           map_location='cpu', weights_only=False)
violence_names = violence_ckpt['class_names']
violence_idx = violence_names.index('violence') if 'violence' in violence_names else 1
violence_model = STGCN(in_ch=3, num_classes=len(violence_names)).to(device).float()
violence_model.load_state_dict(violence_ckpt['model_state_dict'])
violence_model.eval()

print("Loading ST-GCN home-action classifier ...")
action_ckpt = torch.load(f"{WEIGHTS_DIR}/Fine Tuned Weights/home_action_stgcn_best_macrof1.pt",
                         map_location='cpu', weights_only=False)
action_names = action_ckpt['class_names']
action_alert_idx = action_ckpt.get('alert_class_indices', [])
action_model = STGCN(in_ch=3, num_classes=len(action_names)).to(device).float()
action_model.load_state_dict(action_ckpt['model_state_dict'])
action_model.eval()

print("Loading EfficientNet-B0 emotion classifier ...")
emotion_ckpt = torch.load(f"{WEIGHTS_DIR}/Fine Tuned Weights/emotion_fer_best.pt",
                          map_location='cpu', weights_only=False)
emotion_cfg = emotion_ckpt.get('config', {})
emotion_names = emotion_ckpt['class_names']
EMO_SIZE = int(emotion_cfg.get('img_size', 160))
EMO_MEAN = np.array(emotion_cfg.get('mean', [0.485, 0.456, 0.406]), dtype=np.float32)
EMO_STD = np.array(emotion_cfg.get('std', [0.229, 0.224, 0.225]), dtype=np.float32)
emotion_model = timm.create_model(emotion_cfg.get('model_name', 'efficientnet_b0'),
                                  pretrained=False, num_classes=len(emotion_names)).to(device).float()
emotion_model.load_state_dict(emotion_ckpt['model_state_dict'])
emotion_model.eval()

print("Loading InsightFace buffalo_l ...")
face_app = FaceAnalysis(name='buffalo_l', root='/content/insightface')
face_app.prepare(ctx_id=0 if device == 'cuda' else -1, det_size=(640, 640))

print("\nAll models resident in VRAM.")
print("  violence classes:", violence_names)
print("  action classes  :", action_names, "| alert idx:", action_alert_idx)
print("  emotion classes :", emotion_names)

In [ ]:
# Cell 5 — Helpers, smoothing state, and tunables
import supervision as sv
from collections import deque
from base64 import b64decode, b64encode
from PIL import Image
import io, time

tracker = sv.ByteTrack()
FONT = cv2.FONT_HERSHEY_SIMPLEX

# ---- Tunables (raise strides if your runtime is slow; lower for more responsiveness) ----
POSE_IMGSZ      = 512     # full-frame pose resolution
PERSON_IMGSZ    = 640
STRIDE_WEAPON   = 2       # run weapon detector every N frames
STRIDE_FIRE     = 3       # run fire detector every N frames
STRIDE_FACE     = 5       # run face+emotion every N frames
STRIDE_STGCN    = 6       # run ST-GCN action/violence every N frames
ALERT_TTL       = 12      # frames an alert stays visible if not refreshed
BOX_ALPHA       = 0.5     # EMA weight for box corners (prev)
KPT_ALPHA       = 0.5     # EMA weight for keypoints (prev)
CLIP_LEN        = 32
KEYPT_CONF      = 0.35    # joint confidence gate for drawing/heuristics

THREAT_WEIGHTS = {'fire': 1.0, 'smoke': 1.0, 'weapon': 0.90, 'fall': 0.90,
                  'lying': 0.90, 'violence': 0.85, 'aggressive_guard': 0.80,
                  'overcrowd': 0.50, 'cry': 0.35, 'stress': 0.45}

def threat_level(score):
    if score <= 0.30: return 'GREEN', (0, 200, 0)
    if score <= 0.60: return 'YELLOW', (0, 215, 255)
    if score <= 0.80: return 'ORANGE', (0, 140, 255)
    return 'RED', (0, 0, 255)

# ---- Per-track smoothing + history state ----
track_state = {}     # tid -> {'box': np4, 'kpts': (17,3), 'hist': deque, 'emotion': str,
                     #         'stress': float, 'conf': float, 'seen': frame_idx}
active_alerts = {}   # key -> {'text': str, 'w': float, 'ttl': int}
cache = {'weapons': [], 'fires': []}   # last drawn detections (persist between strided runs)
FRAME_IDX = 0
fps_ema = 0.0

_FIREARM = ('gun', 'pistol', 'rifle', 'firearm', 'shotgun', 'handgun', 'weapon')
_KNIFE = ('knife', 'blade', 'dagger', 'machete')

def normalize_weapon(raw):
    n = str(raw).lower()
    if any(t in n for t in _KNIFE):    return 'knife'
    if any(t in n for t in _FIREARM):  return 'firearm'
    return None

def iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0: return 0.0
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])
    return inter / float(areaA + areaB - inter)

def box_containment(kpts, box):
    x1, y1, x2, y2 = box
    in_count = 0
    valid_count = 0
    for kp in kpts:
        if kp[2] > 0.35:
            valid_count += 1
            if x1 <= kp[0] <= x2 and y1 <= kp[1] <= y2:
                in_count += 1
    if valid_count == 0: return 0.0
    return in_count / valid_count

def associate_weapon(wb, persons, pad=0.15, raised_ratio=0.12):
    w_cy = (wb[1] + wb[3]) / 2.0
    best, best_inter = None, 0.0
    for p in persons:
        px1, py1, px2, py2 = p['box']
        pw, ph = px2 - px1, py2 - py1
        ex1, ey1, ex2, ey2 = px1 - pw*pad, py1 - ph*pad, px2 + pw*pad, py2 + ph*pad
        ix = max(0.0, min(wb[2], ex2) - max(wb[0], ex1))
        iy = max(0.0, min(wb[3], ey2) - max(wb[1], ey1))
        inter = ix * iy
        if inter > best_inter:
            best_inter = inter
            raised = (w_cy <= py1 + raised_ratio*ph) or (wb[3] < py1)
            best = {'tid': p['tid'], 'raised': bool(raised)}
    return best

def ear_106(lm, idx):
    p = lm[idx]
    A = np.linalg.norm(p[1] - p[4]); B = np.linalg.norm(p[2] - p[5])
    C = np.linalg.norm(p[0] - p[3])
    return (A + B) / (2.0 * max(C, 1e-6))

def fire_hsv_composite(crop, raw_conf, conf_thr=0.45):
    conf_score = max(0.0, min(1.0, (raw_conf - conf_thr) / max(0.01, 1.0 - conf_thr)))
    hsv_score = 0.40
    if crop.size > 0 and crop.shape[0] > 2 and crop.shape[1] > 2:
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        h, s, v = hsv[:, :, 0].astype(np.float32), hsv[:, :, 1].astype(np.float32), hsv[:, :, 2].astype(np.float32)
        v_mean, v_90 = float(v.mean()), float(np.percentile(v, 90))
        s_mean, s_med = float(s.mean()), float(np.median(s))
        hue_fire = float(np.mean((h <= 30) | (h >= 160)))
        fire = (min(1, max(0, (v_90 - 100)/100))*0.35 + min(1, max(0, (s_mean - 20)/80))*0.30
                + min(1, max(0, (hue_fire - 0.15)/0.45))*0.35)
        smoke = (min(1, max(0, (60 - s_med)/60))*0.55 + min(1, max(0, 1 - abs(v_mean - 140)/100))*0.45)
        hsv_score = max(fire, smoke)
    return 0.5 * conf_score + 0.5 * hsv_score

def _sample_idx(n, clip=CLIP_LEN):
    if n >= clip: return [int(v) for v in np.linspace(0, n - 1, clip)]
    return list(range(n)) + [n - 1] * (clip - n)

def build_clip(hist_lists, h, w, clip=CLIP_LEN):
    M = len(hist_lists)
    out = np.zeros((clip, M, 17, 3), dtype=np.float32)
    for m, hist in enumerate(hist_lists):
        if len(hist) == 0: continue
        idx = _sample_idx(len(hist), clip)
        for t, ii in enumerate(idx):
            p = hist[ii].astype(np.float32).copy()
            p[:, 0] = (p[:, 0] / w - 0.5) * 2.0
            p[:, 1] = (p[:, 1] / h - 0.5) * 2.0
            out[t, m] = p
    return torch.from_numpy(out).permute(3, 0, 2, 1).contiguous().unsqueeze(0)

def add_alert(key, text, weight, ttl=ALERT_TTL):
    active_alerts[key] = {'text': text, 'w': weight, 'ttl': ttl}

def draw_skeleton(frame, kpts):
    for j in range(17):
        if kpts[j][2] > KEYPT_CONF:
            cv2.circle(frame, (int(kpts[j][0]), int(kpts[j][1])), 3, (0, 255, 80), -1)
    for a, b in COCO_EDGES:
        if kpts[a][2] > KEYPT_CONF and kpts[b][2] > KEYPT_CONF:
            cv2.line(frame, (int(kpts[a][0]), int(kpts[a][1])),
                     (int(kpts[b][0]), int(kpts[b][1])), (0, 200, 60), 2)

def draw_alert_panel(frame, level, color, fps, n_people):
    h, w = frame.shape[:2]
    alerts = sorted(active_alerts.values(), key=lambda a: -a['w'])
    lines = [f"FPS {fps:4.1f}   PEOPLE {n_people}"]
    lines += ["- " + a['text'] for a in alerts[:8]]
    fs = max(0.45, min(0.7, w / 1280.0))
    th = 1
    line_h = int(26 * fs / 0.5)
    pad = 10
    widths = [cv2.getTextSize(s, FONT, fs, th)[0][0] for s in lines]
    title = f"THREAT: {level}"
    widths.append(cv2.getTextSize(title, FONT, fs + 0.1, 2)[0][0])
    pw = max(widths) + pad * 2
    title_h = int(line_h + 8)
    ph = title_h + pad + len(lines) * line_h
    x2, y1 = w - 10, 10
    x1, y2 = x2 - pw, y1 + ph
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (25, 25, 25), -1)
    cv2.rectangle(overlay, (x1, y1), (x2, y1 + title_h), color, -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    cv2.putText(frame, title, (x1 + pad, y1 + int(title_h * 0.7)), FONT, fs + 0.1, (255, 255, 255), 2)
    yy = y1 + title_h + line_h - 6
    for i, s in enumerate(lines):
        col = (235, 235, 235) if i == 0 else (210, 230, 255)
        cv2.putText(frame, s, (x1 + pad, yy), FONT, fs, col, th)
        yy += line_h

print("Helpers and live state ready.")

In [ ]:
# Cell 6 — Per-frame pipeline (strided + smoothed for low jitter)
def process_frame(img_b64):
    global FRAME_IDX, fps_ema
    t0 = time.time()
    FRAME_IDX += 1

    header, encoded = img_b64.split(",", 1)
    frame = cv2.cvtColor(np.array(Image.open(io.BytesIO(b64decode(encoded)))), cv2.COLOR_RGB2BGR)
    H, W = frame.shape[:2]

    # 1) Person detection + ByteTrack (every frame)
    pres = person_model(frame, imgsz=PERSON_IMGSZ, classes=[0], conf=0.40, verbose=False)[0]
    tracked = tracker.update_with_detections(sv.Detections.from_ultralytics(pres))
    persons = []
    if tracked.tracker_id is not None:
        for i in range(len(tracked)):
            persons.append({'tid': int(tracked.tracker_id[i]),
                            'box': [int(v) for v in tracked.xyxy[i]],
                            'conf': float(tracked.confidence[i])})

    # 2) Single full-frame pose pass (every frame, FP32) → match to tracks by IoU
    pose_res = pose_model.predict(frame, imgsz=POSE_IMGSZ, classes=[0], conf=0.30,
                                  verbose=False, half=False)[0]
    pose_boxes, pose_kpts = [], []
    if pose_res.keypoints is not None and pose_res.boxes is not None and len(pose_res.boxes) > 0:
        pose_boxes = pose_res.boxes.xyxy.cpu().numpy()
        pose_kpts = pose_res.keypoints.data.cpu().numpy()   # (K,17,3) in frame coords

    live_tids = set()
    for p in persons:
        tid = p['tid']; live_tids.add(tid)
        st = track_state.get(tid)
        if st is None:
            st = {'box': np.array(p['box'], np.float32), 'kpts': None,
                  'hist': deque(maxlen=CLIP_LEN), 'emotion': '', 'stress': 0.0,
                  'conf': p['conf'], 'seen': FRAME_IDX}
            track_state[tid] = st
        # EMA box corners
        st['box'] = BOX_ALPHA * st['box'] + (1 - BOX_ALPHA) * np.array(p['box'], np.float32)
        st['conf'] = p['conf']; st['seen'] = FRAME_IDX
        # nearest pose by box containment
        best_k, best_contain = None, 0.40
        for bi in range(len(pose_kpts)):
            score = box_containment(pose_kpts[bi], p['box'])
            if score > best_contain:
                best_contain, best_k = score, pose_kpts[bi]
        if best_k is not None:
            new_k = best_k.astype(np.float32).copy()
            if st['kpts'] is None:
                st['kpts'] = new_k
            else:
                m = (new_k[:, 2] > KEYPT_CONF)
                sm = st['kpts'].copy()
                sm[m, :2] = KPT_ALPHA * sm[m, :2] + (1 - KPT_ALPHA) * new_k[m, :2]
                sm[:, 2] = new_k[:, 2]
                st['kpts'] = sm
            st['hist'].append(new_k)

    # drop stale tracks
    for tid in [t for t, s in track_state.items() if FRAME_IDX - s['seen'] > 30]:
        track_state.pop(tid, None)

    # 3) Weapon detection (strided) → cache + alert
    if FRAME_IDX % STRIDE_WEAPON == 0:
        cache['weapons'] = []
        wres = weapon_model.predict(frame, conf=0.45, verbose=False, half=True)[0]
        for b in wres.boxes:
            cls = normalize_weapon(weapon_model.names[int(b.cls[0])])
            if cls is None: continue
            wb = [int(v) for v in b.xyxy[0]]; wc = float(b.conf[0])
            assoc = associate_weapon(wb, [{'tid': p['tid'], 'box': p['box']} for p in persons])
            raised = bool(assoc and assoc['raised'])
            # high recall: a single hit (>=0.45) flags; raised => critical
            cache['weapons'].append({'box': wb, 'cls': cls, 'conf': wc, 'raised': raised})

    # 4) Fire/smoke detection (strided) → HSV composite gate → cache + alert
    if FRAME_IDX % STRIDE_FIRE == 0:
        cache['fires'] = []
        fres = fire_model.predict(frame, imgsz=PERSON_IMGSZ, conf=0.50, verbose=False, half=True)[0]
        min_area = 0.001 * H * W
        for b in fres.boxes:
            name = fres.names[int(b.cls[0])].lower()
            fb = [int(v) for v in b.xyxy[0]]; fc = float(b.conf[0])
            if (fb[2]-fb[0]) * (fb[3]-fb[1]) < min_area: continue
            comp = fire_hsv_composite(frame[max(0,fb[1]):fb[3], max(0,fb[0]):fb[2]], fc)
            if comp >= 0.35:
                cache['fires'].append({'box': fb, 'type': name, 'conf': fc})

    # weapons/fires alerts (refresh every frame from cache so panel stays lit)
    for wh in cache['weapons']:
        sev = 'RAISED ' if wh['raised'] else ''
        add_alert(f"weapon_{wh['cls']}", f"{sev}WEAPON: {wh['cls'].upper()} {wh['conf']:.0%}", THREAT_WEIGHTS['weapon'])
    for fe in cache['fires']:
        add_alert(f"fire_{fe['type']}", f"{fe['type'].upper()} {fe['conf']:.0%}", THREAT_WEIGHTS.get(fe['type'], 1.0))

    # 5) Face + emotion (strided) → per-track emotion/stress + EAR cry
    if FRAME_IDX % STRIDE_FACE == 0:
        faces = face_app.get(frame)
        for face in faces:
            fb = [int(v) for v in face.bbox]
            tid, best = None, 0.15
            for p in persons:
                ov = iou(fb, p['box'])
                if ov > best: best, tid = ov, p['tid']
            if tid is None or tid not in track_state: continue
            x1, y1, x2, y2 = max(0, fb[0]), max(0, fb[1]), min(W, fb[2]), min(H, fb[3])
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0 or (x2 - x1) < 20: continue
            rgb = cv2.cvtColor(cv2.resize(crop, (EMO_SIZE, EMO_SIZE)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            xin = torch.from_numpy(((rgb - EMO_MEAN) / EMO_STD)).permute(2, 0, 1).unsqueeze(0).to(device).float()
            with torch.no_grad():
                pr = torch.softmax(emotion_model(xin).float(), -1)[0].cpu().numpy()
            track_state[tid]['emotion'] = emotion_names[int(pr.argmax())]
            sw = {'fear': 1.0, 'angry': 0.7, 'sad': 0.6, 'disgust': 0.4, 'surprise': 0.2}
            track_state[tid]['stress'] = float(sum(pr[i] * sw.get(emotion_names[i], 0.0) for i in range(len(emotion_names))))
            lm = getattr(face, 'landmark', None)
            if lm is not None and lm.shape[0] == 106:
                ear = (ear_106(lm, [35, 37, 38, 39, 41, 40]) + ear_106(lm, [89, 91, 92, 93, 95, 96])) / 2.0
                cry = max(0.0, min(1.0, (0.28 - ear) / 0.11))
                if cry > 0.5:
                    add_alert(f"cry_{tid}", f"CRYING ID{tid} {cry:.0%}", THREAT_WEIGHTS['cry'])

    # 6) Geometric pose heuristics (every frame, cheap) + ST-GCN (strided)
    for p in persons:
        tid = p['tid']; st = track_state.get(tid)
        if st is None or st['kpts'] is None: continue
        k = st['kpts']
        if min(k[5][2], k[6][2], k[11][2], k[12][2]) < 0.4: continue
        msx, msy = (k[5][0] + k[6][0]) / 2, (k[5][1] + k[6][1]) / 2
        mhx, mhy = (k[11][0] + k[12][0]) / 2, (k[11][1] + k[12][1]) / 2
        torso = abs(mhy - msy); sw = abs(k[5][0] - k[6][0])
        spine = np.degrees(np.arctan2(abs(mhx - msx), abs(mhy - msy)))
        if spine > 65 or torso < max(10.0, sw) * 0.5:
            add_alert(f"fall_{tid}", f"FALL ID{tid}", THREAT_WEIGHTS['fall'])
        if len(st['hist']) >= 15:
            arr = np.array(list(st['hist'])[-15:])
            vx = np.var(arr[:, :, 0], axis=0).mean(); vy = np.var(arr[:, :, 1], axis=0).mean()
            swn = max(10.0, sw)
            if vx / swn < 0.05 and vy / swn < 0.05:
                add_alert(f"lying_{tid}", f"UNRESPONSIVE ID{tid}", THREAT_WEIGHTS['lying'])
        if k[9][2] > 0.4 and k[10][2] > 0.4 and k[9][1] < msy and k[10][1] < msy:
            add_alert(f"guard_{tid}", f"AGGRESSIVE GUARD ID{tid}", THREAT_WEIGHTS['aggressive_guard'])

    if FRAME_IDX % STRIDE_STGCN == 0:
        # per-track home action
        for tid, st in list(track_state.items()):
            if len(st['hist']) >= 16:
                x = build_clip([list(st['hist'])], H, W).to(device).float()
                with torch.no_grad():
                    pr = torch.softmax(action_model(x).float(), -1)[0]
                ti = int(pr.argmax()); tp = float(pr[ti])
                if ti in action_alert_idx and tp > 0.65:
                    add_alert(f"act_{tid}", f"{action_names[ti].upper()} ID{tid} {tp:.0%}",
                              THREAT_WEIGHTS.get('lying', 0.9))
        # 2-person violence
        ready = [t for t, s in track_state.items() if len(s['hist']) >= 16 and t in live_tids]
        ready.sort(key=lambda t: -track_state[t]['conf'])
        if len(ready) >= 2:
            hists = [list(track_state[ready[0]]['hist']), list(track_state[ready[1]]['hist'])]
            x = build_clip(hists, H, W).to(device).float()
            with torch.no_grad():
                pr = torch.softmax(violence_model(x).float(), -1)[0]
            vp = float(pr[violence_idx])
            if vp > 0.65:
                add_alert("violence", f"VIOLENCE {vp:.0%}", THREAT_WEIGHTS['violence'])

    # ---- decay alerts ----
    for key in list(active_alerts.keys()):
        active_alerts[key]['ttl'] -= 1
        if active_alerts[key]['ttl'] <= 0:
            active_alerts.pop(key)

    # ---- threat score (max over active alerts + stress) ----
    score = max([a['w'] for a in active_alerts.values()], default=0.0)
    score = max(score, max([s['stress'] for s in track_state.values()], default=0.0) * THREAT_WEIGHTS['stress'])
    if len(persons) >= 10:
        add_alert("overcrowd", f"OVERCROWDING ({len(persons)})", THREAT_WEIGHTS['overcrowd'])
        score = max(score, THREAT_WEIGHTS['overcrowd'])
    level, color = threat_level(score)

    # ---- DRAW ----
    out = frame
    for fe in cache['fires']:
        b = fe['box']
        cv2.rectangle(out, (b[0], b[1]), (b[2], b[3]), (0, 140, 255), 2)
        cv2.putText(out, f"{fe['type'].upper()} {fe['conf']:.0%}", (b[0], b[1] - 6), FONT, 0.5, (0, 140, 255), 2)
    for wh in cache['weapons']:
        b = wh['box']; c = (0, 0, 255); thick = 4 if wh['raised'] else 2
        cv2.rectangle(out, (b[0], b[1]), (b[2], b[3]), c, thick)
        lab = wh['cls'].upper() + (" RAISED!" if wh['raised'] else "")
        cv2.putText(out, f"{lab} {wh['conf']:.0%}", (b[0], b[1] - 6), FONT, 0.5, c, 2)
    for tid, st in track_state.items():
        if st['seen'] != FRAME_IDX: continue
        if st['kpts'] is not None:
            draw_skeleton(out, st['kpts'])
        b = st['box'].astype(int)
        # box color reflects this track's worst active alert
        bc = (0, 220, 0)
        for key, a in active_alerts.items():
            if key.endswith(f"_{tid}") or key == "violence":
                _, ac = threat_level(a['w']); bc = ac if a['w'] >= 0.6 else bc
        cv2.rectangle(out, (b[0], b[1]), (b[2], b[3]), bc, 2)
        lbl = f"ID{tid}"
        if st['emotion']:
            lbl += f" {st['emotion'].upper()} S{st['stress']:.1f}"
        cv2.putText(out, lbl, (b[0], b[1] - 8), FONT, 0.5, bc, 2)

    dt = time.time() - t0
    fps_ema = 0.9 * fps_ema + 0.1 * (1.0 / dt) if dt > 0 else fps_ema
    draw_alert_panel(out, level, color, fps_ema, len(persons))

    ok, buf = cv2.imencode(".jpg", out, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
    return "data:image/jpeg;base64," + b64encode(buf).decode("utf-8")

print("process_frame ready.")

In [ ]:
# Cell 7 — Open the browser webcam and stream to process_frame
from google.colab import output
from IPython.display import HTML

output.register_callback('notebook.run_inference', process_frame)

html_str = '''
<div style="display:flex;flex-direction:column;align-items:center;background:#11151c;padding:22px;border-radius:14px;width:720px;margin:0 auto;font-family:system-ui,sans-serif;box-shadow:0 10px 30px rgba(0,0,0,.5);">
  <h2 style="color:#fff;margin:0 0 4px;font-weight:600;">Vision AI — Live Threat Detection</h2>
  <p style="color:#8a97a8;margin:0 0 16px;font-size:13px;">Webcam → T4 GPU → annotated overlay (boxes, skeletons, top-right alerts)</p>
  <canvas id="cv" width="640" height="480" style="border:3px solid #222b36;border-radius:10px;background:#222b36;"></canvas>
  <div style="margin-top:16px;display:flex;gap:14px;">
    <button id="go" style="padding:11px 22px;font-weight:700;background:#22c55e;color:#fff;border:none;border-radius:6px;cursor:pointer;">Start Camera</button>
    <button id="stop" style="padding:11px 22px;font-weight:700;background:#ef4444;color:#fff;border:none;border-radius:6px;cursor:pointer;" disabled>Stop</button>
  </div>
  <div id="st" style="margin-top:12px;color:#b6c2d1;font-family:monospace;font-size:12px;">Status: idle</div>
</div>
<script>
(async function() {
  const go = document.getElementById('go'), stop = document.getElementById('stop');
  const canvas = document.getElementById('cv'), ctx = canvas.getContext('2d');
  const st = document.getElementById('st');
  let video = null, stream = null, active = false;

  async function initCam() {
    video = document.createElement('video');
    video.autoplay = true; video.playsInline = true;
    stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
    video.srcObject = stream;
    await new Promise(r => video.onloadedmetadata = r);
  }

  async function loop() {
    const hc = document.createElement('canvas'); hc.width = 640; hc.height = 480;
    const hx = hc.getContext('2d');
    while (active) {
      hx.drawImage(video, 0, 0, 640, 480);
      const data = hc.toDataURL('image/jpeg', 0.8);
      try {
        const res = await google.colab.kernel.invokeFunction('notebook.run_inference', [data], {});
        let url = res.data['text/plain'];
        if (url.startsWith("'") || url.startsWith('"')) {
          url = url.slice(1, -1);
        }
        const img = new Image();
        await new Promise((resolve, reject) => {
          img.onload = resolve;
          img.onerror = reject;
          img.src = url;
        });
        ctx.drawImage(img, 0, 0, 640, 480);
      } catch (e) { console.error(e); }
      // pacing: next capture starts only after the previous inference returns (self-throttling)
      await new Promise(r => setTimeout(r, 30));
    }
  }

  go.onclick = async () => {
    go.disabled = true; st.innerText = 'Status: starting camera...';
    try { await initCam(); active = true; stop.disabled = false; st.innerText = 'Status: streaming'; loop(); }
    catch (e) { st.innerText = 'Error: webcam blocked/unavailable'; go.disabled = false; }
  };
  stop.onclick = () => {
    active = false;
    if (stream) stream.getTracks().forEach(t => t.stop());
    go.disabled = false; stop.disabled = true; st.innerText = 'Status: stopped';
    ctx.clearRect(0, 0, 640, 480);
  };
})();
</script>
'''
HTML(html_str)